In [ ]:
import numpy as np
import pandas as pd
import torch
import json

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

from sklearn.metrics import f1_score, precision_score, recall_score

from google.colab import drive
drive.mount("/content/drive", force_remount=True)
OUTPUT_DIR = "/content/drive/MyDrive/RedditSentimentAnalysis/saved_models"
FINAL_MODEL_PATH = "/content/drive/MyDrive/RedditSentimentAnalysis/GoEmotionsFinalModel"

In [ ]:
%pip install transformers datasets accelerate scikit-learn

In [ ]:
dataset = load_dataset("go_emotions")

label_names = dataset["train"].features["labels"].feature.names
num_labels = len(label_names)

print("Number of labels:", num_labels)
print("Labels:", label_names)

In [ ]:
model_name = "cardiffnlp/twitter-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_and_encode(batch):
    tokenized = tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )
    batch_labels = []
    for labels_list in batch["labels"]:
        encoded = np.zeros(num_labels, dtype=np.float32)
        encoded[labels_list] = 1.0
        batch_labels.append(encoded)
    tokenized["labels"] = batch_labels
    return tokenized

dataset = dataset.map(tokenize_and_encode, batched=True, remove_columns=["text", "id"])
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

In [ ]:
# Sqrt-dampened class weighting (matches GoEmotionsImprovedRoBERTa baseline)
# Dampening with sqrt prevents extreme weights for very rare classes,
# which destabilizes training calibration.

labels_matrix = np.array(dataset["train"]["labels"])

pos_counts = labels_matrix.sum(axis=0)
neg_counts = labels_matrix.shape[0] - pos_counts

pos_weight_calc = np.sqrt(neg_counts / (pos_counts + 1e-6))
pos_weight = torch.tensor(pos_weight_calc, dtype=torch.float)

print("Sample weights (first 5):", pos_weight[:5])

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    problem_type="multi_label_classification"
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]

    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)

    return {
        "macro_f1":       f1_score(labels, preds, average="macro",  zero_division=0),
        "micro_f1":       f1_score(labels, preds, average="micro",  zero_division=0),
        "macro_precision": precision_score(labels, preds, average="macro", zero_division=0),
        "macro_recall":    recall_score(labels, preds, average="macro",    zero_division=0),
    }

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=3e-5,
    warmup_ratio=0.1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=6,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=500,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

In [ ]:
class WeightedTrainer(Trainer):
    def __init__(self, pos_weight, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight = pos_weight

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        loss_fct = torch.nn.BCEWithLogitsLoss(
            pos_weight=self.pos_weight.to(logits.device)
        )
        loss = loss_fct(logits, labels.float())
        return (loss, outputs) if return_outputs else loss


trainer = WeightedTrainer(
    pos_weight=pos_weight,
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

trainer.save_model(FINAL_MODEL_PATH)
tokenizer.save_pretrained(FINAL_MODEL_PATH)
print(f"Model saved to {FINAL_MODEL_PATH}")

In [ ]:
# Evaluate on GoEmotions validation set
metrics = trainer.evaluate()
print("Validation metrics:")
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}")

In [ ]:
# Per-class threshold optimization on the GoEmotions validation set.
# Searches 0.30–0.90 per class (floor of 0.30 prevents the 0.10 trap where
# the optimizer predicts every sample for rare classes, destroying Hamming loss).

val_output = trainer.predict(dataset["validation"])
val_logits = val_output.predictions
if isinstance(val_logits, tuple):
    val_logits = val_logits[0]

val_probs  = 1 / (1 + np.exp(-val_logits))
val_labels = val_output.label_ids.astype(int)

optimal_thresholds = np.zeros(num_labels)
threshold_grid = np.arange(0.30, 0.91, 0.05)  # floor at 0.30

print(f"{'Emotion':<22} {'Threshold':>9}  {'F1':>6}  {'Recall':>7}  {'Precision':>9}  {'Support':>7}")
print("-" * 70)

for i, name in enumerate(label_names):
    best_f1, best_t = 0.0, 0.5
    for t in threshold_grid:
        preds = (val_probs[:, i] >= t).astype(int)
        f = f1_score(val_labels[:, i], preds, zero_division=0)
        if f > best_f1:
            best_f1 = f
            best_t  = t
    optimal_thresholds[i] = best_t

    final_preds = (val_probs[:, i] >= best_t).astype(int)
    rec  = recall_score(val_labels[:, i], final_preds, zero_division=0)
    prec = precision_score(val_labels[:, i], final_preds, zero_division=0)
    sup  = int(val_labels[:, i].sum())
    print(f"{name:<22} {best_t:>9.3f}  {best_f1:>6.3f}  {rec:>7.3f}  {prec:>9.3f}  {sup:>7}")

threshold_path = FINAL_MODEL_PATH + "/optimal_thresholds.json"
with open(threshold_path, "w") as f:
    json.dump({name: float(t) for name, t in zip(label_names, optimal_thresholds)}, f, indent=2)

print(f"\nThresholds saved to {threshold_path}")

In [ ]:
# Verify: apply optimal thresholds and compare macro F1/F2 vs fixed 0.5
fixed_preds   = (val_probs >= 0.5).astype(int)
optimal_preds = (val_probs >= optimal_thresholds).astype(int)

print("Fixed 0.5 threshold:")
print(f"  macro F1 : {f1_score(val_labels, fixed_preds, average='macro',  zero_division=0):.4f}")
print(f"  macro F2 : {fbeta_score(val_labels, fixed_preds, beta=2, average='macro', zero_division=0):.4f}")
print(f"  macro rec: {recall_score(val_labels, fixed_preds, average='macro', zero_division=0):.4f}")

print("\nPer-class optimal thresholds:")
print(f"  macro F1 : {f1_score(val_labels, optimal_preds, average='macro',  zero_division=0):.4f}")
print(f"  macro F2 : {fbeta_score(val_labels, optimal_preds, beta=2, average='macro', zero_division=0):.4f}")
print(f"  macro rec: {recall_score(val_labels, optimal_preds, average='macro', zero_division=0):.4f}")

In [ ]:
# Utility: run inference on any text using the saved optimal thresholds

def predict_emotions(text, threshold_override=None):
    """
    Returns a dict of {emotion: probability} for emotions that exceed their
    per-class optimal threshold.
    Pass threshold_override (float) to use a single global cutoff instead.
    """
    thresholds = optimal_thresholds if threshold_override is None \
                 else np.full(num_labels, threshold_override)

    inputs = tokenizer(text, return_tensors="pt", truncation=True,
                       padding=True, max_length=128)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = model(**inputs).logits

    probs = torch.sigmoid(logits).cpu().numpy()[0]

    return {
        label_names[i]: round(float(probs[i]), 4)
        for i in range(num_labels)
        if probs[i] >= thresholds[i]
    }


# Quick sanity checks
tests = [
    "I really hate what this government is doing, it's absolutely disgusting.",
    "I hope thousands of Americans are contacting their congressional representatives!",
    "Just reading the news today.",
]
for t in tests:
    print(f"Text: {t!r}")
    print(f"  => {predict_emotions(t)}\n")